# Experiment: Opacity Rationale Clustering

Objective:
- Cluster opacity rationales in `exp0.3/results.csv` using the same family of embedding and projection choices reported for Clio.
- Produce reusable artifacts: rationale rows, cached embeddings, cluster assignments, a cluster summary table, and an interactive UMAP scatter.


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import HTML

from interviewer.analysis.opacity_rationale_clustering import run_opacity_rationale_analysis

SEED = 7
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_CSV = PROJECT_ROOT / "outputs/runs/exp0.3/results.csv"
OUTPUT_DIR = PROJECT_ROOT / "artifacts/opacity_rationale_clustering"
RESULTS_CSV, OUTPUT_DIR


## Plan

- Use `sentence-transformers/all-mpnet-base-v2`, the sentence-transformer Anthropic reports for Clio.
- Cluster only `potential` and `clear` rationales unless you want to explicitly include `none` rows.
- Use Clio-style UMAP defaults: `n_neighbors=15`, `min_dist=0`, `metric='cosine'`.


In [ ]:
# Parameters
analysis_kwargs = {
    "output_dir": OUTPUT_DIR,
    "mechanisms": None,
    "levels": ("potential", "clear"),
    "forms": None,
    "sample_size": None,
    "embedding_model": "sentence-transformers/all-mpnet-base-v2",
    "embedding_batch_size": 64,
    "embedding_device": None,
    "n_clusters": 12,
    "random_state": SEED,
    "umap_neighbors": 15,
    "umap_min_dist": 0.0,
    "force_reembed": False,
}
analysis_kwargs


## Results

Run the full pipeline. The first run will download the sentence-transformer model if it is not already cached locally.

In [ ]:
artifacts = run_opacity_rationale_analysis(RESULTS_CSV, **analysis_kwargs)
artifacts


In [ ]:
cluster_summary = pd.read_csv(artifacts.cluster_summary_path)
clustered = pd.read_csv(artifacts.clustered_points_path)

cluster_summary.head(10)


In [ ]:
clustered.groupby(["cluster_label", "mechanism_label"]).size().unstack(fill_value=0)


In [ ]:
HTML(filename=str(artifacts.figure_path))


## Next steps

- Sweep `n_clusters` and compare whether clusters become mechanism-specific or genuinely cross-mechanism.
- Try separate runs for `production` versus `avoidance` rationales.
- If you want to align even more tightly with Clio, cluster model-generated summaries of rationales rather than the raw rationale strings.
